# Understanding the Data Engine Interface World

## Introduction

This investigation begins from a different boundary from the one taken in the earlier notebook.

There, the emphasis was on following the internal authoring path of the Data Engine: the datasets, artefacts, policies, and event surfaces that participate in the staged creation of the governed world. Here, the engine is treated as a sealed black box. Our concern is no longer with how the engine internally authored that world step by step, but with the exposed governed data world that has already been handed to downstream consumers through the Data Interface Pack.

The posture we are now taking is that of an advanced data analytics specialist working inside a live fraud platform company. The platform is assumed to be operational, teams are already working across ingestion, real-time decisioning, offline learning, observability, governance, and case operations, and we have been given a governed operating extract covering January to March. This is the world available to us for investigation.

That means our first job is not yet to perform deep statistical analysis on a particular dataset. Our first job is to orient ourselves in the exposed platform-facing data world: what assets have been handed to us through the interface pack, what role each one plays in the live operating platform, which outputs represent traffic, which represent context, which represent truth products, which represent audit evidence, and which exist only as gate or operational artefacts. In other words, before analyzing the data itself, we need to understand how the exposed data world is arranged around the operating life of the platform.

This matters because not every surface is available to every part of the platform at every moment. Some outputs are relevant to ingestion and traffic handling. Some are only meaningful once context joins are applied. Some belong to offline truth and case reconstruction rather than live decisioning. Some are audit or telemetry surfaces that help explain or verify what happened but are not themselves canonical business traffic. So part of this investigation is to map the interface-facing assets to the operating responsibilities and decision contexts of the live system.

The practical question guiding the opening of this notebook is therefore:

> Given the governed outputs exposed by the Data Interface Pack, what data world has the platform actually handed to downstream analytical consumers, and how does that world map onto the operating parts of the live fraud system?

Only after that mapping is clear will we move into deeper analytical investigation of the datasets themselves.


## Mapping the Exposed Interface World to the Live Platform

Before we inspect individual datasets, we first need a platform-facing orientation of the kinds of assets the interface pack has exposed to us.

At this stage, the point is not yet to study the internal statistical properties of any one dataset. The point is to understand what kind of operating role each exposed asset class plays in the live system. In a working platform, not every surface is being used the same way. Some are the traffic itself. Some are the context that must be joined onto that traffic. Some are truth products that only become available after the fact. Some are audit or telemetry outputs that support explanation, debugging, or governance rather than business operations directly.

The interface pack already gives us a black-box taxonomy for this, and that taxonomy is our first orientation lens:

- `traffic_primitives`: event skeletons or early traffic-like structures that are not yet the platform's canonical business traffic.
- `behavioural_streams`: the canonical synthetic production traffic that is eligible to move through ingestion, event handling, and downstream feature consumption.
- `behavioural_context`: join surfaces that provide the surrounding context needed to interpret or enrich behavioural streams.
- `truth_products`: offline truth, labels, case, and bank-view products that are used for supervision, evaluation, and investigative reconstruction rather than live traffic handling.
- `audit_evidence`: replay, RNG, selection, and trace evidence that helps explain how the governed world was produced or how a run behaved, but is not itself business traffic.
- `ops_telemetry`: operational run journals and supporting run-time evidence used for monitoring, validation, and debugging.
- `gate_artifacts`: validation bundles, indexes, `_passed.flag` files, and receipts that determine whether downstream readers are even allowed to treat a surface as authoritative.

From a live-platform point of view, these classes map onto different operating zones:

| Interface-pack class | Live-platform meaning |
|---|---|
| `traffic_primitives` | upstream event structures or skeletons that may support traffic building but are not yet the canonical stream seen by the platform as business traffic |
| `behavioural_streams` | the closest thing to live transaction/event traffic available to ingestion, event-bus, and feature-consumption paths |
| `behavioural_context` | supporting join context used by ingestion, RTDL, and feature planes to make behavioural streams interpretable and enrichable |
| `truth_products` | offline outcomes and reconstructed truths used later by analytics, model evaluation, supervision, fraud review, and case-management workflows |
| `audit_evidence` | explanatory and forensic evidence used to understand what the governed system did, why it did it, and whether it can be replayed or defended |
| `ops_telemetry` | operational observability evidence used to monitor state runs, diagnose failures, and understand runtime behaviour |
| `gate_artifacts` | readiness controls that determine whether an exposed output is cleared for authoritative downstream use |

This means that when we begin exploring the interface-world datasets themselves, we should not read them as one flat pile of outputs. We should read them according to their role in the operating life of the platform.

The immediate goal of this opening section is therefore to build a mental map of the exposed world using three questions:

1. What outputs exist in the interface pack and in our pinned run?
2. What operating role does each output play in the live platform?
3. At what stage of platform use would that output actually become available or meaningful to a downstream team?

Once this mapping is in place, we can then move from the abstract interface taxonomy into the concrete output families that our analytical team has actually received.


## Workbench Mapping Anchor

A workbench mapping of the downstream-facing interface world has now been completed and is recorded here:

- [Interface-pack asset mapping](../watson_workbench/notes/interface_world/interface_pack_asset_mapping.md)
- [Downstream-estate inventory](../watson_workbench/exports/interface_world/interface_downstream_estate_present_only.csv)
- [Downstream-estate summary](../watson_workbench/exports/interface_world/interface_downstream_estate_summary.json)

This mapping keeps the black-box posture strict. It excludes the hidden world-building side of the engine and keeps our attention on the operating estate the interface pack actually hands to downstream consumers.

Under that boundary, the received downstream estate is a narrow `5B` / `6B` slice made up of exactly `17` present outputs in this pinned run:

- `1` traffic primitive
- `2` behavioural streams
- `4` behavioural-context surfaces
- `4` truth products
- `6` gate artefacts

At the live edge, the real traffic front door is still the same seven surfaces:

- `arrival_events_5B`
- `s1_arrival_entities_6B`
- `s1_session_index_6B`
- `s2_event_stream_baseline_6B`
- `s2_flow_anchor_baseline_6B`
- `s3_event_stream_with_fraud_6B`
- `s3_flow_anchor_with_fraud_6B`

Around that front door, the downstream team is also given the four `s4_*` truth products and the `5B` / `6B` validation bundles that authorize reads from those surfaces.

So for the rest of this notebook, our practical reading lens is now much tighter:

1. traffic primitive
2. behavioural streams
3. behavioural context
4. truth products
5. read-authorizing gate artefacts

Equally important is what falls out of scope for this black-box investigation: the world-building layers `1A`, `1B`, `2A`, `2B`, `3A`, `3B`, `5A`, and `6A`. Those may be visible in the run tree, but they are not part of the received operating estate we are trying to understand here.
